# 03 · Model Evaluation and Analysis

Evaluate the humanitarian `all_data_en` classification models
(TF-IDF + Logistic Regression and BERT).

This notebook:

- Evaluates model performance using accuracy, precision, recall, and F1-score
- Compares performance across different crisis events
- Generates confusion matrices and classification reports
- Identifies and analyzes incorrectly classified posts
- Saves evaluation results and reports in `results/`

In [9]:
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

print("Evaluation libraries imported successfully!")

Evaluation libraries imported successfully!


In [4]:
from pathlib import Path

ROOT = Path.cwd().resolve()

if not (ROOT / "data").is_dir():
    ROOT = ROOT.parent

DATA = ROOT / "data/processed"
RESULTS = ROOT / "results"

RESULTS.mkdir(parents=True, exist_ok=True)

print("Project root:", ROOT)
print("Processed data:", DATA)
print("Results folder:", RESULTS)

Project root: /workspaces/cs1684-crisis-nlp
Processed data: /workspaces/cs1684-crisis-nlp/data/processed
Results folder: /workspaces/cs1684-crisis-nlp/results


In [5]:
print("Available processed files:")

for file in sorted(DATA.iterdir()):
    print(file.name)

Available processed files:
humanitarian_dev.tsv
humanitarian_test.tsv
humanitarian_train.tsv
preprocessing_class_counts.csv
preprocessing_removed_rows.tsv
preprocessing_report.json
preprocessing_split_counts.csv


## 1. Load Processed Data

Load the preprocessed humanitarian datasets to examine their structure and prepare for model evaluation.

In [10]:
train_df = pd.read_csv(DATA / "humanitarian_train.tsv", sep="\t")
dev_df = pd.read_csv(DATA / "humanitarian_dev.tsv", sep="\t")
test_df = pd.read_csv(DATA / "humanitarian_test.tsv", sep="\t")

print("Training samples:", len(train_df))
print("Validation samples:", len(dev_df))
print("Testing samples:", len(test_df))

print("\nDataset columns:")
print(test_df.columns.tolist())

print("\nFirst 5 testing samples:")
display(test_df.head())

Training samples: 61164
Validation samples: 8934
Testing samples: 17356

Dataset columns:
['id', 'event', 'source', 'text', 'lang', 'lang_conf', 'class_label', 'split', 'source_row', 'text_clean', 'event_group', 'possible_encoding_issue']

First 5 testing samples:


,id,event,source,text,lang,lang_conf,class_label,split,source_row,text_clean,event_group,possible_encoding_issue
0,21767,disaster_events,drd-figureeight-multimedia,Staff at our feeding centre say chronic malnou...,en,1.0,requests_or_needs,test,1,Staff at our feeding centre say chronic malnou...,disaster_events,False
1,324921639701716993,2013_west_texas,crisislext6,@johnnyjeff05 you comin down for the summer se...,en,1.0,not_humanitarian,test,2,USER you comin down for the summer semesters r...,2013_west_texas,False
2,262348997677686785,2012_sandy_hurricane-ontopic,crisislext6,@Nooram_M yea it's upstate I'm like a few hour...,en,1.0,not_humanitarian,test,3,USER yea it's upstate I'm like a few hours away.,2012_sandy_hurricane,False
3,508341453676756992,2014_pakistan_floods,crisisnlp-cf,RT @GDLanglands: Teach every Pakistani that it...,en,1.0,not_humanitarian,test,4,RT USER : Teach every Pakistani that it is not...,2014_pakistan_floods,True
4,540037166374866944,2014_philippines_typhoon,crisisnlp-cf,RT @TedWinnerCNN: Stay with @CNN for live cvg ...,en,1.0,displaced_and_evacuations,test,5,RT USER : Stay with USER for live cvg as Typho...,2014_philippines_typhoon,True
